In [11]:
import duckdb
import pandas as pd
import re
from pathlib import Path

RAW = Path('../data/raw')
con = duckdb.connect()
events_path = RAW / 'userid-timestamp-artid-artname-traid-traname.tsv'

In [13]:
sessionized = con.execute(f"""
    SELECT
        user_id,
        timestamp,
        artist_name,
        track_name,
        LEAD(timestamp) OVER (PARTITION BY user_id ORDER BY timestamp) AS next_timestamp,
        DATE_DIFF('second', timestamp,
            LEAD(timestamp) OVER (PARTITION BY user_id ORDER BY timestamp)) AS gap_seconds,
        CASE
            WHEN DATE_DIFF('second',
                LAG(timestamp) OVER (PARTITION BY user_id ORDER BY timestamp),
                timestamp) > 1200
            OR LAG(timestamp) OVER (PARTITION BY user_id ORDER BY timestamp) IS NULL
            THEN 1 ELSE 0
        END AS is_new_session
    FROM read_csv_auto('{events_path}', delim='\t', header=False,
        names=['user_id', 'timestamp', 'artist_id', 'artist_name', 'track_id', 'track_name'])
    ORDER BY user_id, timestamp
""").df()

sessionized['session_id'] = sessionized.groupby('user_id')['is_new_session'].cumsum()
sessionized['is_skip'] = (sessionized['gap_seconds'] < 30) & (sessionized['gap_seconds'].notna())

print(sessionized.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(19150868, 9)


In [15]:
sessionized.to_parquet('../data/processed/sessionized_events.parquet', index=False)

In [16]:
features = pd.read_csv('../data/raw/spotify_audio_features.csv')
features = features.drop(columns=['Unnamed: 0'])

def normalize(text):
    text = str(text).lower().strip()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

unique_lastfm = sessionized[['artist_name', 'track_name']].drop_duplicates().reset_index(drop=True)
unique_lastfm['artist_norm'] = unique_lastfm['artist_name'].apply(normalize)
unique_lastfm['track_norm'] = unique_lastfm['track_name'].apply(normalize)
features['artist_norm'] = features['artists'].apply(normalize)
features['track_norm'] = features['track_name'].apply(normalize)

In [17]:
matched_exact = unique_lastfm.merge(
    features[['artist_norm', 'track_norm', 'danceability', 'energy', 'valence',
              'tempo', 'acousticness', 'loudness', 'popularity', 'track_genre']],
    on=['artist_norm', 'track_norm'],
    how='left',
    indicator=True
)

play_counts = sessionized.groupby(['artist_name', 'track_name']).size().reset_index(name='play_count')
matched_exact = matched_exact.merge(play_counts, on=['artist_name', 'track_name'], how='left')

play_coverage = matched_exact.groupby('_merge')['play_count'].sum()
print(play_coverage)
print(f"\nPlay-level coverage from exact match: {play_coverage['both'] / play_coverage.sum():.1%}")

C:\Users\91809\AppData\Local\Temp\ipykernel_36200\3777640671.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  play_coverage = matched_exact.groupby('_merge')['play_count'].sum()


_merge
left_only     17810446
right_only           0
both          11737338
Name: play_count, dtype: int64

Play-level coverage from exact match: 39.7%


In [18]:
# unmatched tracks, sorted by how much listening volume they represent
unmatched = matched_exact[matched_exact['_merge'] == 'left_only'].sort_values('play_count', ascending=False)

# how many tracks do we need to fuzzy-match to recover, say, 80% of the missing volume?
unmatched['cumulative_share'] = unmatched['play_count'].cumsum() / unmatched['play_count'].sum()
tracks_needed_for_80pct = (unmatched['cumulative_share'] <= 0.80).sum()

print(f"Total unmatched tracks: {len(unmatched):,}")
print(f"Tracks needed to cover 80% of missing play volume: {tracks_needed_for_80pct:,}")
unmatched.head(15)

Total unmatched tracks: 1,480,431
Tracks needed to cover 80% of missing play volume: 262,307


,artist_name,track_name,artist_norm,track_norm,danceability,energy,valence,tempo,acousticness,loudness,popularity,track_genre,_merge,play_count,cumulative_share
11692,The Postal Service,Such Great Heights,the postal service,such great heights,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,3992,0.000224
18650,Boy Division,Love Will Tear Us Apart,boy division,love will tear us apart,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,3663,0.000430
20122,Muse,Supermassive Black Hole,muse,supermassive black hole,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,3483,0.000625
11980,Death Cab For Cutie,Soul Meets Body,death cab for cutie,soul meets body,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,3479,0.000821
20121,Muse,Starlight,muse,starlight,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,3060,0.000993
12183,Arcade Fire,Rebellion (Lies),arcade fire,rebellion lies,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,3048,0.001164
28864,Interpol,Evil,interpol,evil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,2989,0.001331
133765,Kanye West,Love Lockdown,kanye west,love lockdown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,2950,0.001497
20133,Muse,Time Is Running Out,muse,time is running out,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,2945,0.001662
11231,Bloc Party,Banquet,bloc party,banquet,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,2906,0.001826


In [19]:
# does Radiohead appear in the Spotify features file at all, under any track?
features[features['artist_norm'] == 'radiohead'][['artists', 'track_name', 'track_genre']]

,artists,track_name,track_genre
2053,Radiohead,Creep,alt-rock
2055,Radiohead,No Surprises,alt-rock
2406,Radiohead,Karma Police,alt-rock
2462,Radiohead,High and Dry,alt-rock
2519,Radiohead,Fake Plastic Trees,alt-rock
2610,Radiohead,Exit Music (For A Film),alt-rock
2612,Radiohead,Weird Fishes/ Arpeggi,alt-rock
2704,Radiohead,Nude,alt-rock
2813,Radiohead,Paranoid Android,alt-rock
2853,Radiohead,How to Disappear Completely,alt-rock


In [20]:
# how many distinct genres does this Spotify file actually contain, and how many tracks per genre?
print(features['track_genre'].nunique())
print(features.groupby('track_genre').size().describe())

114
count     114.0
mean     1000.0
std         0.0
min      1000.0
25%      1000.0
50%      1000.0
75%      1000.0
max      1000.0
dtype: float64


In [29]:
def fuzzy_match_row(row, threshold=85):
    artist_match, artist_score, _ = process.extractOne(
        row['artist_norm'], spotify_artists_list, scorer=fuzz.ratio
    ) or (None, 0, None)

    if artist_score < threshold:
        return pd.Series([None, None, artist_score, None])

    candidate_tracks = spotify_by_artist[artist_match]
    track_match, track_score, _ = process.extractOne(
        row['track_norm'], candidate_tracks, scorer=fuzz.ratio
    ) or (None, 0, None)

    if track_score < threshold:
        return pd.Series([artist_match, None, artist_score, track_score])

    return pd.Series([artist_match, track_match, artist_score, track_score])

In [ ]:
from rapidfuzz import process, fuzz

# only fuzzy-match tracks worth the computational cost (top ~80% of missing volume)
to_fuzzy_match = unmatched.iloc[:tracks_needed_for_80pct].copy()

# build a lookup: for each unique normalized artist in the Spotify file, what tracks exist under it
spotify_by_artist = features.groupby('artist_norm')['track_norm'].apply(list).to_dict()
spotify_artists_list = list(spotify_by_artist.keys())

def fuzzy_match_row(row, threshold=85):
    # first, find the closest artist name match
    artist_match, artist_score, _ = process.extractOne(
        row['artist_norm'], spotify_artists_list, scorer=fuzz.ratio
    ) or (None, 0, None)

    if artist_score < threshold:
        return None  # artist itself doesn't look like anything in Spotify's file

    # then, within that artist's tracks, find the closest track name match
    candidate_tracks = spotify_by_artist[artist_match]
    track_match, track_score, _ = process.extractOne(
        row['track_norm'], candidate_tracks, scorer=fuzz.ratio
    ) or (None, 0, None)

    if track_score < threshold:
        return None

    return pd.Series([artist_match, track_match, artist_score, track_score])

to_fuzzy_match[['matched_artist', 'matched_track', 'artist_score', 'track_score']] = \
    to_fuzzy_match.apply(fuzzy_match_row, axis=1)

fuzzy_success_rate = to_fuzzy_match['matched_track'].notna().mean()
print(f"Fuzzy match recovered: {fuzzy_success_rate:.1%} of the {len(to_fuzzy_match):,} attempted tracks")
to_fuzzy_match[to_fuzzy_match['matched_track'].notna()].head(10)